In [ ]:

import pandas as pd
from pathlib import Path


# ==========================================
# 1. CONFIGURATION
# ==========================================

ATTENDANCE_FILE = Path(
    "output/second_reports/attendance_feature_engineered.xlsx"
)

CALENDAR_FILE = Path(
    "output/dim_calendar.parquet"
)

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DAILY_PARQUET = OUTPUT_DIR / "fact_attendance_daily.parquet"
DAILY_CSV = OUTPUT_DIR / "fact_attendance_daily.csv"

WEEKLY_PARQUET = OUTPUT_DIR / "fact_employee_week.parquet"
WEEKLY_CSV = OUTPUT_DIR / "fact_employee_week.csv"


# ==========================================
# 2. LOAD DATA
# ==========================================

print("Loading attendance data...")

if not ATTENDANCE_FILE.exists():
    raise FileNotFoundError(
        f"Attendance file not found: {ATTENDANCE_FILE}"
    )

if not CALENDAR_FILE.exists():
    raise FileNotFoundError(
        f"Calendar file not found: {CALENDAR_FILE}"
    )

attendance = pd.read_excel(ATTENDANCE_FILE)

calendar = pd.read_parquet(CALENDAR_FILE)

print("Attendance records:", len(attendance))
print("Calendar records:", len(calendar))


# ==========================================
# 3. VALIDATE REQUIRED COLUMNS
# ==========================================

attendance_required = [
    "employee_id",
    "attendance_date",
    "cost_center_code",
    "cost_center_type",
    "weekly_work_quota",
    "schedule_type",
    "clock_in_time",
    "clock_out_time",
    "clock_in_time_be",
    "clock_out_time_be"
]

calendar_required = [
    "attendance_date",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]


missing_attendance = [
    col for col in attendance_required
    if col not in attendance.columns
]

missing_calendar = [
    col for col in calendar_required
    if col not in calendar.columns
]

if missing_attendance:
    raise ValueError(
        f"Missing attendance columns: {missing_attendance}"
    )

if missing_calendar:
    raise ValueError(
        f"Missing calendar columns: {missing_calendar}"
    )


# ==========================================
# 4. STANDARDIZE DATA TYPES
# ==========================================

attendance["employee_id"] = (
    attendance["employee_id"]
    .astype("string")
    .str.strip()
)

attendance["attendance_date"] = (
    pd.to_datetime(
        attendance["attendance_date"],
        errors="coerce"
    )
    .dt.normalize()
)

calendar["attendance_date"] = (
    pd.to_datetime(
        calendar["attendance_date"],
        errors="coerce"
    )
    .dt.normalize()
)

calendar["week_start"] = (
    pd.to_datetime(
        calendar["week_start"],
        errors="coerce"
    )
    .dt.normalize()
)

calendar["week_end"] = (
    pd.to_datetime(
        calendar["week_end"],
        errors="coerce"
    )
    .dt.normalize()
)

# Validate employee and attendance date

if attendance["employee_id"].isna().any():
    raise ValueError(
        "Missing employee_id found."
    )

if attendance["employee_id"].eq("").any():
    raise ValueError(
        "Empty employee_id found."
    )

if attendance["attendance_date"].isna().any():
    raise ValueError(
        "Invalid attendance_date found."
    )

if calendar["attendance_date"].isna().any():
    raise ValueError(
        "Invalid calendar attendance_date found."
    )

if calendar["attendance_date"].duplicated().any():
    raise ValueError(
        "Duplicate dates found in calendar."
    )


# ==========================================
# 5. PREPARE ATTENDANCE FLAGS
# ==========================================

# Original clock OR Before Editing (BE)
# determines whether clock activity exists.


def check_clock_value(series):

    value = (
        series.astype("string")
        .str.strip()
        .str.upper()
    )

    invalid_values = [
        "",
        "NULL",
        "NONE",
        "NAN",
        "NAT",
        "-",
        "<NA>"
    ]

    return (
        series.notna()
        &
        value.ne("").fillna(False)
        &
        ~value.isin(invalid_values).fillna(False)
    )


# Individual clock flags

attendance["has_clock_in"] = (
    check_clock_value(
        attendance["clock_in_time"]
    )
    |
    check_clock_value(
        attendance["clock_in_time_be"]
    )
)

attendance["has_clock_out"] = (
    check_clock_value(
        attendance["clock_out_time"]
    )
    |
    check_clock_value(
        attendance["clock_out_time_be"]
    )
)

# Combined attendance activity

attendance["has_attendance_activity"] = (
    attendance["has_clock_in"]
    |
    attendance["has_clock_out"]
)


# ==========================================
# 6. VALIDATE COST CENTER CLASSIFICATION
# ==========================================

valid_cost_center_types = [
    "FIXED_5",
    "NON_FIXED_6"
]

valid_quota_mapping = {
    "FIXED_5": 5,
    "NON_FIXED_6": 6
}

attendance["cost_center_type"] = (
    attendance["cost_center_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

attendance["schedule_type"] = (
    attendance["schedule_type"]
    .astype("string")
    .str.strip()
    .str.upper()
)

attendance["weekly_work_quota"] = pd.to_numeric(
    attendance["weekly_work_quota"],
    errors="coerce"
).astype("Int64")

# Verify classification and quota consistency

for cost_type, expected_quota in valid_quota_mapping.items():

    invalid_quota = (
        attendance["cost_center_type"].eq(cost_type)
        &
        attendance["weekly_work_quota"].ne(expected_quota)
    )

    if invalid_quota.any():
        raise ValueError(
            f"Incorrect quota for {cost_type}. "
            f"Expected {expected_quota}."
        )

invalid_types = (
    ~attendance["cost_center_type"].isin(
        valid_cost_center_types + ["UNKNOWN"]
    )
)

if invalid_types.any():
    raise ValueError(
        "Unexpected cost_center_type values found."
    )


# ==========================================
# 7. REMOVE DUPLICATE EMPLOYEE-DATE RECORDS
# ==========================================

# Daily fact grain:
# One employee = one attendance date.

def consistent_value(series):

    values = series.dropna().unique()

    if len(values) == 1:
        return values[0]

    return pd.NA


daily = (
    attendance.groupby(
        ["employee_id", "attendance_date"],
        as_index=False,
        dropna=False
    )
    .agg(

        attendance_record_count=(
            "attendance_date",
            "size"
        ),

        has_clock_in=(
            "has_clock_in",
            "max"
        ),

        has_clock_out=(
            "has_clock_out",
            "max"
        ),

        has_attendance_activity=(
            "has_attendance_activity",
            "max"
        ),

        cost_center_code=(
            "cost_center_code",
            consistent_value
        ),

        cost_center_type=(
            "cost_center_type",
            consistent_value
        ),

        weekly_work_quota=(
            "weekly_work_quota",
            consistent_value
        ),

        schedule_type=(
            "schedule_type",
            consistent_value
        ),

        store_type=(
            "store_type",
            consistent_value
        )
    )
)

print("\nDaily aggregation completed.")

print("Original rows:", len(attendance))
print("Daily rows:", len(daily))
print("Duplicate rows consolidated:", len(attendance) - len(daily))


# ==========================================
# 8. HANDLE COST CENTER CONFLICTS
# ==========================================

# If one employee has conflicting cost center
# classifications on the same date, do not
# assign an assumed quota.

daily["cost_center_type"] = (
    daily["cost_center_type"]
    .fillna("UNKNOWN")
)

daily["schedule_type"] = (
    daily["schedule_type"]
    .fillna("UNKNOWN")
)

daily["weekly_work_quota"] = (
    daily["weekly_work_quota"]
    .astype("Int64")
)

unknown_daily = (
    daily["cost_center_type"] == "UNKNOWN"
)

daily.loc[
    unknown_daily,
    "weekly_work_quota"
] = pd.NA

daily.loc[
    unknown_daily,
    "schedule_type"
] = "UNKNOWN"


# ==========================================
# 9. MERGE DAILY ATTENDANCE WITH CALENDAR
# ==========================================

calendar_columns = [
    "attendance_date",
    "week_start",
    "week_end",
    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",
    "weekday_national_holidays",
    "weekday_joint_leaves"
]

calendar_reference = calendar[
    calendar_columns
].copy()

# Attendance dates must exist in calendar

calendar_dates = set(
    calendar_reference["attendance_date"]
)

attendance_dates = set(
    daily["attendance_date"]
)

unmatched_dates = (
    attendance_dates - calendar_dates
)

if unmatched_dates:

    raise ValueError(
        "Attendance dates not found in calendar: "
        f"{sorted(unmatched_dates)[:10]}"
    )

daily = daily.merge(
    calendar_reference,
    on="attendance_date",
    how="left",
    validate="many_to_one"
)

# Standardize calendar flags

daily["is_national_holiday"] = (
    daily["is_national_holiday"]
    .fillna(False)
    .astype(bool)
)

daily["is_joint_leave"] = (
    daily["is_joint_leave"]
    .fillna(False)
    .astype(bool)
)

daily["is_weekend"] = (
    daily["is_weekend"]
    .fillna(False)
    .astype(bool)
)


# ==========================================
# 10. DAILY WORK REQUIREMENT
# ==========================================

# FIXED_5:
# Monday-Friday, excluding national holidays.
#
# NON_FIXED_6:
# Actual working day is UNKNOWN.


daily["is_expected_workday"] = pd.Series(
    pd.NA,
    index=daily.index,
    dtype="boolean"
)

is_fixed = (
    daily["cost_center_type"] == "FIXED_5"
)

is_weekday = ~daily["is_weekend"]

is_fixed_workday = (
    is_fixed
    &
    is_weekday
    &
    ~daily["is_national_holiday"]
)

# FIXED_5 expected workday

daily.loc[
    is_fixed,
    "is_expected_workday"
] = is_fixed_workday[is_fixed]


# Daily work requirement

daily["daily_work_requirement"] = pd.Series(
    pd.NA,
    index=daily.index,
    dtype="Int64"
)

daily.loc[
    is_fixed_workday,
    "daily_work_requirement"
] = 1

daily.loc[
    is_fixed & ~is_fixed_workday,
    "daily_work_requirement"
] = 0


# ==========================================
# 11. CREATE WEEKLY CALENDAR REFERENCE
# ==========================================

# Use calendar as the source of weekly
# holiday information.

weekly_calendar = (
    calendar.groupby("week_start")
    .agg(

        week_end=(
            "week_end",
            "max"
        ),

        calendar_days=(
            "attendance_date",
            "nunique"
        ),

        weekday_national_holidays=(
            "weekday_national_holidays",
            "max"
        ),

        weekday_joint_leaves=(
            "weekday_joint_leaves",
            "max"
        )
    )
    .reset_index()
)

# A complete week must contain seven dates.
# First and last weeks of the calendar year
# may be incomplete.

weekly_calendar["is_full_calendar_week"] = (
    weekly_calendar["calendar_days"] == 7
)

# Calendar coverage warning

print("\nCALENDAR WEEK VALIDATION")

print(
    weekly_calendar[
        [
            "week_start",
            "calendar_days",
            "is_full_calendar_week"
        ]
    ].to_string(index=False)
)


# ==========================================
# 12. AGGREGATE EMPLOYEE WEEKLY ATTENDANCE
# ==========================================

# Weekly fact grain:
# One employee = one week.


def count_true(series):

    return int(series.fillna(False).sum())


weekly = (
    daily.groupby(
        ["employee_id", "week_start"],
        as_index=False,
        dropna=False
    )
    .agg(

        cost_center_code=(
            "cost_center_code",
            consistent_value
        ),

        cost_center_type=(
            "cost_center_type",
            consistent_value
        ),

        weekly_work_quota=(
            "weekly_work_quota",
            consistent_value
        ),

        schedule_type=(
            "schedule_type",
            consistent_value
        ),

        store_type=(
            "store_type",
            consistent_value
        ),

        attendance_record_count=(
            "attendance_record_count",
            "sum"
        ),

        attendance_days_observed=(
            "attendance_date",
            "nunique"
        ),

        actual_attendance_days=(
            "has_attendance_activity",
            count_true
        ),

        actual_clock_in_days=(
            "has_clock_in",
            count_true
        ),

        actual_clock_out_days=(
            "has_clock_out",
            count_true
        ),

        actual_expected_workdays_attended=(
            "is_expected_workday",
            lambda x: count_true(
                x
                &
                daily.loc[x.index, "has_attendance_activity"]
            )
        ),

        attendance_outside_expected_days=(
            "attendance_date",
            lambda x: int(
                (
                    daily.loc[x.index, "has_attendance_activity"]
                    &
                    is_fixed.loc[x.index]
                    &
                    ~daily.loc[x.index, "is_expected_workday"].fillna(False)
                ).sum()
            )
        )
    )
)


# ==========================================
# 13. MERGE WEEKLY CALENDAR
# ==========================================

weekly = weekly.merge(
    weekly_calendar,
    on="week_start",
    how="left",
    validate="many_to_one"
)

if weekly["calendar_days"].isna().any():
    raise ValueError(
        "Some employee weeks have no calendar reference."
    )


# ==========================================
# 14. CALCULATE WEEKLY REQUIRED DAYS
# ==========================================

# Incomplete calendar weeks:
# Do not calculate weekly quota gap because
# the full seven-day period is unavailable.

weekly["weekly_required_days"] = pd.Series(
    pd.NA,
    index=weekly.index,
    dtype="Int64"
)

full_week = weekly["is_full_calendar_week"]

fixed_mask = (
    weekly["cost_center_type"] == "FIXED_5"
)

non_fixed_mask = (
    weekly["cost_center_type"] == "NON_FIXED_6"
)

# FIXED_5:
# Five days minus weekday national holidays.

fixed_required = (
    5 - weekly["weekday_national_holidays"]
)

if (fixed_required < 0).any():
    raise ValueError(
        "Weekly national holidays exceed fixed work quota."
    )

weekly.loc[
    full_week & fixed_mask,
    "weekly_required_days"
] = fixed_required[
    full_week & fixed_mask
].astype("int64")

# NON_FIXED_6:
# Always six days, regardless of holidays.

weekly.loc[
    full_week & non_fixed_mask,
    "weekly_required_days"
] = 6


# ==========================================
# 15. CALCULATE ATTENDANCE GAP
# ==========================================

weekly["attendance_gap"] = pd.Series(
    pd.NA,
    index=weekly.index,
    dtype="Int64"
)

# FIXED_5:
# Compare attendance on expected workdays.

fixed_gap = (
    weekly["weekly_required_days"]
    -
    weekly["actual_expected_workdays_attended"]
)

weekly.loc[
    full_week & fixed_mask,
    "attendance_gap"
] = fixed_gap[
    full_week & fixed_mask
]

# NON_FIXED_6:
# Compare actual attendance days with quota.
#
# This is a quota gap, NOT proof of absence
# or an attendance violation.

non_fixed_gap = (
    weekly["weekly_required_days"]
    -
    weekly["actual_attendance_days"]
)

weekly.loc[
    full_week & non_fixed_mask,
    "attendance_gap"
] = non_fixed_gap[
    full_week & non_fixed_mask
]


# ==========================================
# 16. CREATE WEEKLY ATTENDANCE STATUS
# ==========================================

weekly["weekly_attendance_status"] = "UNKNOWN"

# Incomplete calendar week

weekly.loc[
    ~full_week,
    "weekly_attendance_status"
] = "PARTIAL_WEEK_NOT_EVALUATED"

# Unknown cost center

unknown_mask = (
    weekly["cost_center_type"] == "UNKNOWN"
)

weekly.loc[
    full_week & unknown_mask,
    "weekly_attendance_status"
] = "UNKNOWN_COST_CENTER"


# FIXED_5

weekly.loc[
    full_week
    & fixed_mask
    & (weekly["attendance_gap"] <= 0),
    "weekly_attendance_status"
] = "QUOTA_MET_OR_EXCEEDED"

weekly.loc[
    full_week
    & fixed_mask
    & (weekly["attendance_gap"] > 0),
    "weekly_attendance_status"
] = "BELOW_REQUIRED_DAYS"


# NON_FIXED_6

weekly.loc[
    full_week
    & non_fixed_mask
    & (weekly["attendance_gap"] <= 0),
    "weekly_attendance_status"
] = "AT_OR_ABOVE_WEEKLY_QUOTA"

weekly.loc[
    full_week
    & non_fixed_mask
    & (weekly["attendance_gap"] > 0),
    "weekly_attendance_status"
] = "BELOW_WEEKLY_QUOTA_REVIEW"


# ==========================================
# 17. CREATE ADDITIONAL WEEKLY INDICATORS
# ==========================================

weekly["is_quota_gap"] = (
    weekly["attendance_gap"] > 0
).astype("boolean")

weekly["is_quota_met"] = (
    weekly["attendance_gap"] <= 0
).astype("boolean")

# These indicators are unknown for partial weeks
# and unknown cost center.

not_evaluated = (
    ~full_week | unknown_mask
)

weekly.loc[
    not_evaluated,
    ["is_quota_gap", "is_quota_met"]
] = pd.NA


# ==========================================
# 18. DATA VALIDATION
# ==========================================

print("\n==========================================")
print("PROCESS 10 DATA VALIDATION")
print("==========================================")


# Daily grain validation

if daily.duplicated(
    ["employee_id", "attendance_date"]
).any():

    raise ValueError(
        "Duplicate employee-date records found."
    )


# Weekly grain validation

if weekly.duplicated(
    ["employee_id", "week_start"]
).any():

    raise ValueError(
        "Duplicate employee-week records found."
    )


# Validate actual attendance days

if (
    weekly["actual_attendance_days"]
    >
    weekly["attendance_days_observed"]
).any():

    raise ValueError(
        "Actual attendance days exceed observed days."
    )


# Validate quota

if (
    weekly.loc[
        weekly["cost_center_type"] == "FIXED_5",
        "weekly_required_days"
    ].dropna() > 5
).any():

    raise ValueError(
        "FIXED_5 quota exceeds five days."
    )

if (
    weekly.loc[
        weekly["cost_center_type"] == "NON_FIXED_6",
        "weekly_required_days"
    ].dropna() != 6
).any():

    raise ValueError(
        "NON_FIXED_6 quota must remain six days."
    )


print("Daily grain: VALID")
print("Weekly grain: VALID")
print("Attendance days: VALID")
print("Weekly quota: VALID")


# ==========================================
# 19. REORDER DAILY COLUMNS
# ==========================================

daily_columns = [
    "employee_id",
    "attendance_date",

    "cost_center_code",
    "cost_center_type",
    "weekly_work_quota",
    "schedule_type",
    "store_type",

    "week_start",
    "week_end",

    "has_clock_in",
    "has_clock_out",
    "has_attendance_activity",

    "is_weekend",
    "is_national_holiday",
    "is_joint_leave",

    "is_expected_workday",
    "daily_work_requirement",

    "attendance_record_count"
]

daily = daily[daily_columns]


# ==========================================
# 20. REORDER WEEKLY COLUMNS
# ==========================================

weekly_columns = [
    "employee_id",

    "week_start",
    "week_end",

    "cost_center_code",
    "cost_center_type",
    "weekly_work_quota",
    "schedule_type",
    "store_type",

    "calendar_days",
    "is_full_calendar_week",

    "weekday_national_holidays",
    "weekday_joint_leaves",

    "weekly_required_days",

    "attendance_record_count",
    "attendance_days_observed",

    "actual_attendance_days",
    "actual_clock_in_days",
    "actual_clock_out_days",

    "actual_expected_workdays_attended",
    "attendance_outside_expected_days",

    "attendance_gap",
    "weekly_attendance_status",

    "is_quota_gap",
    "is_quota_met"
]

weekly = weekly[weekly_columns]


# ==========================================
# 21. EXPORT DAILY AND WEEKLY FACT TABLES
# ==========================================

daily.to_parquet(
    DAILY_PARQUET,
    index=False
)

daily.to_csv(
    DAILY_CSV,
    index=False,
    encoding="utf-8-sig"
)

weekly.to_parquet(
    WEEKLY_PARQUET,
    index=False
)

weekly.to_csv(
    WEEKLY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ==========================================
# 22. SUMMARY REPORT
# ==========================================

print("\n==========================================")
print("PROCESS 10 COMPLETED")
print("WEEKLY ATTENDANCE BUSINESS RULES")
print("==========================================")


print("\nDAILY FACT")

print("Total daily records:", len(daily))


print("\nWEEKLY FACT")

print("Total employee-week records:", len(weekly))


print("\nCOST CENTER DISTRIBUTION")

print(
    weekly["cost_center_type"].value_counts(
        dropna=False
    )
)


print("\nWEEKLY REQUIRED DAYS DISTRIBUTION")

print(
    weekly["weekly_required_days"].value_counts(
        dropna=False
    ).sort_index()
)


print("\nWEEKLY ATTENDANCE STATUS")

print(
    weekly["weekly_attendance_status"].value_counts(
        dropna=False
    )
)


print("\nACTUAL ATTENDANCE DAYS")

print(
    weekly["actual_attendance_days"].describe()
)


print("\nATTENDANCE GAP")

print(
    weekly["attendance_gap"].describe()
)


print("\nPARTIAL CALENDAR WEEKS")

print(
    weekly["is_full_calendar_week"]
    .value_counts(dropna=False)
)


print("\nOUTPUT FILES")

print("Daily Parquet:", DAILY_PARQUET)
print("Daily CSV:", DAILY_CSV)
print("Weekly Parquet:", WEEKLY_PARQUET)
print("Weekly CSV:", WEEKLY_CSV)
print("\n==========================================")
print("PROCESS 10 SUCCESSFULLY FINISHED")
print("==========================================")


Loading attendance data...
Attendance records: 27711
Calendar records: 365

Daily aggregation completed.
Original rows: 27711
Daily rows: 27711
Duplicate rows consolidated: 0

CALENDAR WEEK VALIDATION
week_start  calendar_days  is_full_calendar_week
2025-12-29              4                  False
2026-01-05              7                   True
2026-01-12              7                   True
2026-01-19              7                   True
2026-01-26              7                   True
2026-02-02              7                   True
2026-02-09              7                   True
2026-02-16              7                   True
2026-02-23              7                   True
2026-03-02              7                   True
2026-03-09              7                   True
2026-03-16              7                   True
2026-03-23              7                   True
2026-03-30              7                   True
2026-04-06              7                   True
2026-04-13     